In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))

import config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv(config.DB_LOCATION)

C:\Users\bnpar\AppData\Local\Temp\ipykernel_22724\1970705316.py:12: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(config.DB_LOCATION)


In [2]:
#reduced dataframe
red_df = df[['Name', 'Sex', 'Division', 'BodyweightKg', 'WeightClassKg','Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', 'TotalKg', 'Date', 'Sanctioned', 'Equipment', 'Event']]

### Cleaning dataset



In [3]:
def get_mixed_cols(dataframe):
    all_types_count = {} #nested dict w column heading as outer key. Inner key-value pairs are data types/how many entries with that datatyoe
    for col in dataframe.columns:
        col_types = dataframe[col].map(type)   #applies type(x) to every x in the column so you get series ['string', 'string', 'float'....]
        all_types_count[col] = col_types.value_counts().to_dict()
    mixed_type_cols = []
    for col in all_types_count:
        no_types = len(all_types_count[col].values())
        #we are only appending columns with mixed data types
        if no_types > 1:
            mixed_type_cols.append(col)
    return mixed_type_cols

mixed_cols = get_mixed_cols(red_df)
print(mixed_cols)

['Division', 'WeightClassKg']


In [4]:
#checking if all the floats are because of NaN
for col in mixed_cols:
    mask = red_df[col].map(type) == float
    float_col = red_df.loc[mask, col]
    float_col = float_col.dropna()
    print(float_col)

Series([], Name: Division, dtype: object)
Series([], Name: WeightClassKg, dtype: object)


- All float datatypes are because of NaN entries in WeightClassKg or Division. Want to drop any entries where we dont know what the weight class is.
- Are okay with not knowing the Division i.e. whether the lifter is a Junior or Open lifter for now, as a lifter of any age can set an Open WR.
- Chosen to drop any unsanctioned meets.
- Flitered for Raw powerlifting only (specified in 'Equipment' column).
- Turned dates into datetime objects.
- Filtering for full power events (e.g. excluding bench only comps)

In [5]:
red_df = red_df[red_df['Equipment'] == 'Raw'].copy()
red_df['Date'] = pd.to_datetime(red_df['Date'] , format = '%Y-%m-%d')
red_df['Year'] = red_df['Date'].dt.year
red_df = red_df.dropna(axis = 'rows', subset = ['WeightClassKg'])
unsanctioned = red_df[red_df['Sanctioned'] == 'No'].index.to_list()
red_df = red_df.drop(labels = unsanctioned, axis = 'index')
full_power = red_df[red_df['Event'] == 'SBD'].copy()
#bench_only = red_df[red_df['Event']== 'B'].copy()


Next we flag entries in the dataset that exceed the current world official world record. Note that the total being more than the official WR does not necessarily mean the entry is incorrect. Official WRs can only be set at international meets and lifters may have exceeded this at national or local events, without setting a new official WR.

Also note that the 53kg and 43kg in the men's and women's categories respectively are only in the Junior age division.


In [6]:
m_off_wr_totals = {'53': 559, '59': 669.5, '66': 770, '74': 843, '83': 890, '93': 918, '105': 975.5, '120': 978.5, '120+': 1152.5}
f_off_wr_totals = {'43': 349.5, '47': 435, '52': 482.5, '57': 520, '63': 565, '69': 628, '76': 620, '84': 647, '84+': 737.5}

exceeds_wr = pd.DataFrame(columns = red_df.columns.to_list())

for weight_class in m_off_wr_totals:
    weight_class_entries = full_power[(full_power['WeightClassKg'] == weight_class) & (full_power['Sex'] == 'M')].copy()
    exceeds_class_wr = weight_class_entries[weight_class_entries['TotalKg']>m_off_wr_totals[weight_class] ]
    exceeds_wr = pd.concat([exceeds_wr, exceeds_class_wr])

for weight_class in f_off_wr_totals:
    weight_class_entries = full_power[(full_power['WeightClassKg'] == weight_class) & (full_power['Sex'] == 'F')].copy()
    exceeds_class_wr = weight_class_entries[weight_class_entries['TotalKg']>f_off_wr_totals[weight_class]]
    exceeds_wr = pd.concat([exceeds_wr, exceeds_class_wr])
    
exceeds_wr

C:\Users\bnpar\AppData\Local\Temp\ipykernel_22724\1112979477.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  exceeds_wr = pd.concat([exceeds_wr, exceeds_class_wr])


,Name,Sex,Division,BodyweightKg,WeightClassKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Date,Sanctioned,Equipment,Event,Year
891163,Sergey Fedosienko,M,Open,58.50,59,226.5,170.5,273.0,670.0,2015-12-24,Yes,Raw,SBD,2015
907319,Sergey Fedosienko,M,Open,59.00,59,228.0,167.5,275.0,670.5,2020-03-18,Yes,Raw,SBD,2020
1242982,Joseph Borenstein,M,MR-O,82.91,83,305.0,218.0,377.0,900.0,2025-04-03,Yes,Raw,SBD,2025
1237528,William Ball #2,M,MR-Jr,92.15,93,322.5,222.5,375.0,920.0,2025-09-27,Yes,Raw,SBD,2025
1243024,Jonathan Cayco,M,MR-O,92.85,93,310.0,252.0,357.5,919.5,2025-04-03,Yes,Raw,SBD,2025
1243068,Anthony McNaughton,M,MR-O,103.74,105,365.0,245.0,369.0,979.0,2025-04-03,Yes,Raw,SBD,2025
1241212,Bobb Matthews,M,MR-O,110.75,120,367.5,247.5,395.5,1010.5,2025-11-22,Yes,Raw,SBD,2025
1243132,Rondel Hunte,M,MR-G,119.58,120,375.0,260.0,375.0,1010.0,2025-04-03,Yes,Raw,SBD,2025
1241213,Jesus Olivares,M,MR-O,182.40,120+,478.5,265.0,410.0,1153.5,2025-11-22,Yes,Raw,SBD,2025
937711,Joy Nnamani,F,FR-O,56.70,57,185.0,100.0,235.5,520.5,2025-12-05,Yes,Raw,SBD,2025


All entries that exceed the current world record are credible and correct against the results from those meets.

### Dealing with weight class changes

Next we look for weight classes in the df which are not the weight classes that the IPF and its affiliates currrently use. 


In [7]:
current_f_class_mask = (full_power['WeightClassKg'].isin(f_off_wr_totals.keys())) & (full_power['Sex'] == 'F')
current_m_class_mask = (full_power['WeightClassKg']).isin(m_off_wr_totals.keys()) & (full_power['Sex'] == 'M')

old_class_entries = full_power[(~current_f_class_mask) & (~current_m_class_mask) ].copy()
f_old_class_entries = full_power[~current_f_class_mask].copy()
m_old_class_entries = full_power[~current_m_class_mask].copy()

current_f_classes = full_power[current_f_class_mask].copy()
current_m_classes = full_power[current_m_class_mask].copy()

f'{(len(old_class_entries)/len(full_power))*100 :.2f}% of the dataset is entries that do not match the IPF\'s current weight classes'

"7.08% of the dataset is entries that do not match the IPF's current weight classes"

Given that only 7.08% of the dataset are entries which do not match the IPF's current weight classes, these will be excluded from the dataset. Need to understand the impact of this so will determine when the weight class changes took place. There is no official documentation that describes the weight class changes. So need to look at the dataset to determine when these took place.

In [12]:
f_annual_per_current_class = current_f_classes.groupby(['WeightClassKg', 'Year']).size().reset_index(name = 'AnnualClassParticipation')
total_annual = full_power[full_power['Sex']== 'F'].groupby('Year').size().reset_index(name = 'AnnualParticipation') #annual participation incl 'incurrent' classes

f_annual_per_current_class = f_annual_per_current_class.merge(total_annual, on = 'Year', how = 'left', validate = 'm:1') 

#now we want annual participation excl incurrent classes
current_class_share = f_annual_per_current_class.groupby('Year')['AnnualClassParticipation'].sum().reset_index(name = 'currentClassParticipation')

f_annual_per_current_class = f_annual_per_current_class.merge(current_class_share, how = 'left', on = 'Year')
f_annual_per_current_class['currentClassShare'] = (f_annual_per_current_class['currentClassParticipation']/f_annual_per_current_class['AnnualParticipation'])*100

In [11]:
df_sorted = df.sort_values('Year')
plt.plot(df_sorted['Year'],df_sorted['currentClassShare'], marker = 'o')
plt.title("Percenatge of entries made up by current IPF weight classes by year")
plt.ylabel("current Class Share (%)")
plt.show()

KeyError: 'Year'

Can see that weight classes changed between 2020 and 2021. Need to detect what that weight class shift was